# The AI Trust Gap
## AI Usage, Trust, and Human–AI Decision Making

This notebook analyzes respondent-level data from the **Stack Overflow Developer Survey 2025** to examine relationships between AI usage intensity, trust in AI accuracy, professional experience, management responsibility, AI-agent adoption, and perceived ability of AI to handle complex tasks.

### Research question
**How is trust in AI associated with usage intensity, professional context, and perceived AI capability—and what does this imply for human oversight and AI-assisted decision making?**

### Important boundary
This is an observational survey analysis. The results identify **associations, not causal effects**. Trust is also not the same as objective decision quality.


## 1. Setup


In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## 2. Data source

The analysis uses the public respondent-level **Stack Overflow Developer Survey 2025** dataset.

For reproducibility, place the survey `results.csv` file in the same folder as this notebook, or update `DATA_PATH` below.

The public repository does not need to include the raw survey dataset.


In [ ]:
DATA_PATH = Path("results.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "results.csv was not found. Download the 2025 Stack Overflow Developer Survey "
        "dataset and place results.csv next to this notebook, or update DATA_PATH."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Rows: {len(raw):,}")
print(f"Columns: {len(raw.columns):,}")


## 3. Select relevant variables


In [ ]:
candidate_columns = [
    "ResponseId",
    "AISelect",
    "AIAcc",
    "AIComplex",
    "AIAgents",
    "WorkExp",
    "ICorPM",
    "Country",
]

available = [c for c in candidate_columns if c in raw.columns]
missing = [c for c in candidate_columns if c not in raw.columns]

print("Available columns:", available)
print("Missing columns:", missing)

df = raw[available].copy()


## 4. Data preparation

Professional experience is grouped into three broad bands.

Trust is classified explicitly:

- **Trust** = Highly trust / Somewhat trust
- **Neutral** = Neither trust nor distrust
- **Distrust** = Highly distrust / Somewhat distrust

This explicit mapping avoids accidental substring-based misclassification.


In [ ]:
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")
    df["experience_band"] = pd.cut(
        df["WorkExp"],
        bins=[-np.inf, 5, 10, np.inf],
        labels=["Early career (≤5)", "Mid career (6–10)", "Experienced (10+)"]
    )

def classify_trust(value):
    if pd.isna(value):
        return np.nan
    if value in {"Highly trust", "Somewhat trust"}:
        return "Trust"
    if value == "Neither trust nor distrust":
        return "Neutral"
    if value in {"Highly distrust", "Somewhat distrust"}:
        return "Distrust"
    return np.nan

df["trust_group"] = df["AIAcc"].apply(classify_trust)

df["trust_group"].value_counts(dropna=False)


## 5. Overall trust distribution


In [ ]:
trust_counts = df["trust_group"].value_counts()
trust_pct = (trust_counts / trust_counts.sum() * 100).round(1)

overall_trust = pd.DataFrame({
    "Respondents": trust_counts,
    "Percent": trust_pct
}).reindex(["Trust", "Neutral", "Distrust"])

overall_trust


In [ ]:
ax = overall_trust["Percent"].plot(kind="bar", figsize=(7, 4), legend=False)
ax.set_title("Overall trust in AI accuracy")
ax.set_ylabel("Percent of answered responses")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Segment-analysis helper

Each analysis reports within-group percentages, a chi-square test, and Cramér's V.

Because this is a large sample, **effect size is emphasized over p-value alone**.


In [ ]:
TRUST_ORDER = ["Trust", "Neutral", "Distrust"]

def association_table(data, segment, trust_col="trust_group"):
    subset = data[[segment, trust_col]].dropna()

    counts = pd.crosstab(subset[segment], subset[trust_col])
    counts = counts.reindex(columns=TRUST_ORDER, fill_value=0)

    pct = counts.div(counts.sum(axis=1), axis=0) * 100

    chi2, p, dof, expected = chi2_contingency(counts)

    n = counts.to_numpy().sum()
    r, k = counts.shape
    cramers_v = math.sqrt((chi2 / n) / max(1, min(r - 1, k - 1)))

    return counts, pct.round(1), chi2, p, cramers_v


## 7. AI usage intensity vs trust


In [ ]:
use_counts, use_pct, chi2, p, v = association_table(df, "AISelect")

display(use_pct)
print(f"N = {use_counts.to_numpy().sum():,}")
print(f"Chi-square = {chi2:,.1f}")
print(f"p-value = {p:.3g}")
print(f"Cramér's V = {v:.3f}")


In [ ]:
plot_order = [
    x for x in [
        "Yes, I use AI tools daily",
        "Yes, I use AI tools weekly",
        "Yes, I use AI tools monthly or infrequently",
        "No, but I plan to soon",
        "No, and I don't plan to",
    ]
    if x in use_pct.index
]

ax = use_pct.loc[plot_order, "Trust"].plot(kind="bar", figsize=(10, 5))
ax.set_title("Trust in AI accuracy by AI usage intensity")
ax.set_ylabel("Trust (%)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


### Interpretation

AI usage intensity has a much stronger association with trust than professional experience does.

This does **not** establish whether using AI increases trust or whether people who already trust AI are more likely to use it.


## 8. Professional experience vs trust


In [ ]:
exp_counts, exp_pct, chi2, p, v = association_table(df, "experience_band")

display(exp_pct)
print(f"N = {exp_counts.to_numpy().sum():,}")
print(f"Chi-square = {chi2:,.1f}")
print(f"p-value = {p:.3g}")
print(f"Cramér's V = {v:.3f}")


In [ ]:
ax = exp_pct["Trust"].plot(kind="bar", figsize=(7, 4))
ax.set_title("Trust in AI accuracy by professional experience")
ax.set_ylabel("Trust (%)")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


### Interpretation

Experience differences are statistically detectable, but the effect size is very small.

This is an important correction to a tempting but oversimplified narrative that experienced professionals are substantially more skeptical of AI.


## 9. People managers vs individual contributors


In [ ]:
manager_counts, manager_pct, chi2, p, v = association_table(df, "ICorPM")

display(manager_pct)
print(f"N = {manager_counts.to_numpy().sum():,}")
print(f"Chi-square = {chi2:,.1f}")
print(f"p-value = {p:.3g}")
print(f"Cramér's V = {v:.3f}")


### Interpretation

Management responsibility has a small association with trust. People managers are somewhat more trusting than individual contributors, but the effect remains modest.


## 10. AI-agent adoption vs trust


In [ ]:
agent_counts, agent_pct, chi2, p, v = association_table(df, "AIAgents")

display(agent_pct)
print(f"N = {agent_counts.to_numpy().sum():,}")
print(f"Chi-square = {chi2:,.1f}")
print(f"p-value = {p:.3g}")
print(f"Cramér's V = {v:.3f}")


In [ ]:
agent_order = [
    x for x in [
        "Yes, I use AI agents at work daily",
        "Yes, I use AI agents at work weekly",
        "Yes, I use AI agents at work monthly or infrequently",
        "No, but I plan to soon",
        "No, and I don't plan to",
    ]
    if x in agent_pct.index
]

if agent_order:
    ax = agent_pct.loc[agent_order, "Trust"].plot(kind="bar", figsize=(10, 5))
    ax.set_title("Trust in AI accuracy by AI-agent adoption")
    ax.set_ylabel("Trust (%)")
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


### Interpretation

AI-agent adoption is strongly associated with trust. Respondents using agents more frequently tend to report substantially higher trust in AI accuracy.


## 11. Perceived complex-task capability vs trust


In [ ]:
complex_counts, complex_pct, chi2, p, v = association_table(df, "AIComplex")

display(complex_pct)
print(f"N = {complex_counts.to_numpy().sum():,}")
print(f"Chi-square = {chi2:,.1f}")
print(f"p-value = {p:.3g}")
print(f"Cramér's V = {v:.3f}")


In [ ]:
complex_order = [
    x for x in [
        "Very poor at handling complex tasks",
        "Poor at handling complex tasks",
        "Neither good nor poor at handling complex tasks",
        "Good at handling complex tasks",
        "Very well at handling complex tasks",
    ]
    if x in complex_pct.index
]

if complex_order:
    ax = complex_pct.loc[complex_order, "Trust"].plot(kind="bar", figsize=(10, 5))
    ax.set_title("Trust in AI accuracy by perceived complex-task capability")
    ax.set_ylabel("Trust (%)")
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


### Interpretation

Perceived complex-task capability has the strongest observed association with trust in this Phase 1 analysis.

However, both measures capture related attitudes toward AI, so this relationship should not be interpreted as independent causal evidence.


## 12. Country comparison


In [ ]:
country_counts = (
    df.dropna(subset=["Country", "trust_group"])
      .groupby("Country")
      .size()
)

eligible_countries = country_counts[country_counts >= 200].index

country_subset = df[df["Country"].isin(eligible_countries)]

country_trust = (
    pd.crosstab(country_subset["Country"], country_subset["trust_group"], normalize="index")
      .reindex(columns=TRUST_ORDER, fill_value=0)
      * 100
).round(1)

country_trust.sort_values("Trust", ascending=False).head(20)


Country comparisons are descriptive only. Differences may reflect respondent composition, industry mix, experience, adoption patterns, and survey participation.


## 13. Summary of effect sizes


In [ ]:
summary = []

for label, segment in [
    ("AI usage intensity", "AISelect"),
    ("Professional experience", "experience_band"),
    ("Manager vs IC", "ICorPM"),
    ("AI-agent adoption", "AIAgents"),
    ("Complex-task capability", "AIComplex"),
]:
    counts, pct, chi2, p, v = association_table(df, segment)
    summary.append({
        "Analysis": label,
        "N": int(counts.to_numpy().sum()),
        "Cramer's V": round(v, 3)
    })

effect_sizes = pd.DataFrame(summary).sort_values("Cramer's V", ascending=False)

effect_sizes


In [ ]:
ax = effect_sizes.set_index("Analysis")["Cramer's V"].sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

ax.set_title("Association strength across Phase 1 analyses")
ax.set_xlabel("Cramér's V")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 14. Key findings

The Phase 1 evidence supports five conclusions:

1. **Overall trust is lower than distrust** among respondents who answered the AI-accuracy question.
2. **AI usage intensity is substantially associated with trust.**
3. **AI-agent adoption is strongly associated with trust.**
4. **Perceived ability to handle complex tasks is the strongest observed trust signal.**
5. **Professional experience has only a very small association with trust.**

The stronger research story is therefore not simply that experienced professionals are more skeptical.

> **Trust rises sharply with AI exposure and perceived capability—but observational survey data cannot tell us whether that greater trust improves objective decision quality or mainly increases confidence and willingness to delegate.**


## 15. Limitations

- Cross-sectional survey data cannot establish causal direction.
- Trust and AI usage are self-reported.
- Stack Overflow respondents are not representative of all knowledge workers.
- Perceived complex-task capability and trust are conceptually related.
- Large samples make statistical significance easy to obtain.
- Phase 1 does **not** measure objective decision quality.


## 16. Phase 2 research direction

### Research question
**Does higher AI reliance improve objective decision quality, or mainly increase confidence and willingness to delegate?**

A controlled pilot can compare:

1. **No AI**
2. **AI-assisted**
3. **AI-first**

Potential outcomes include objective decision quality, decision time, confidence, evidence utilization, risk recognition, verification behavior, and the ability to detect and correct an imperfect AI recommendation.


## 17. Portfolio takeaway

Phase 1 identifies an important management problem:

**AI adoption and AI trust move together, but adoption metrics alone cannot show whether human judgment is improving.**

That distinction matters for organizations deploying generative AI in project management, strategy, operations, and knowledge work.
